In [ ]:
#r "BoSSSpad.dll"
using System;
using System.Data;
using System.Collections.Generic;
using System.Linq;
using System.IO;
using ilPSP;
using ilPSP.Utils;
using BoSSS.Platform;
using BoSSS.Foundation;
using BoSSS.Foundation.XDG;
using BoSSS.Foundation.Grid;
using BoSSS.Foundation.Grid.Classic;
using BoSSS.Foundation.IO;
using BoSSS.Solution;
using BoSSS.Solution.Control;
using BoSSS.Solution.GridImport;
using BoSSS.Solution.Statistic;
using BoSSS.Solution.Utils;
using BoSSS.Solution.AdvancedSolvers;
using BoSSS.Solution.Gnuplot;
using BoSSS.Application.BoSSSpad;
using BoSSS.Application.XNSE_Solver;
using BoSSS.Application.XNSFE_Solver;
using static BoSSS.Application.BoSSSpad.BoSSSshell;
using System.Diagnostics;
using Microsoft.AspNetCore.Html;
using System.Text.RegularExpressions;

Init();

A capillary wave with a certain wavelength is placed in a periodic domain.  
The capillary timestep is calculated based on this fixed wavelength.  
The calculations are repeated with this limiting timestep and smaller values.

Finally, it is evaluated wether the simulation is able to run for a complete period of the capillary wave  
or if it crashes.  

The reasoning is the following: Because the capillary timestep restriction (based on grid size) is violated for all of these simulations,  
they will crash at some point. However by exactly defining an intital pertubation in form of a capillary wave,  
the severity of the timestep restriction can be assessed.  

CapillaryWave : $\Delta t_i = 5/(5+i)\Delta t$ Only one full period, $y_0 = 0.01/k$


In [ ]:
string proj = "CapillaryWave";
BoSSSshell.WorkflowMgm.Init(proj);
BoSSSshell.WorkflowMgm.SetNameBasedSessionJobControlCorrelation();


In [77]:
BoSSSshell.WorkflowMgm.Sessions

In [78]:
//BoSSSshell.WorkflowMgm.ResetProject(true, true, true, true);

In [79]:
int[] kelemSeq = new int[]{3};
var GridSeq = new IGridInfo[kelemSeq.Length];

In [ ]:
for(int iGrid = 0; iGrid < GridSeq.Length; iGrid++) {
    
    int kelem = kelemSeq[iGrid];
    
    GridCommons grd;       
    double[] xNodes = GenericBlas.Linspace(-1.0, 1.0, 2);
    double[] yNodes = GenericBlas.Linspace(-1.0, 1.0, 2); 
    double h = (yNodes.Last()-yNodes.First())/kelem;    
    var box1 = new GridCommons.GridBox(new double[] {xNodes.First(), yNodes.First()}, new double[] {xNodes.Last(), yNodes.Last()}, kelem, kelem);
    var box2 = new GridCommons.GridBox(new double[] {xNodes.First(), yNodes.First()+h}, new double[] {xNodes.Last(), yNodes.Last()-h}, 3*kelem, kelem);
    var box3 = new GridCommons.GridBox(new double[] {xNodes.First(), yNodes.First()+h+h/3}, new double[] {xNodes.Last(), yNodes.Last()-h-h/3}, 3*3*kelem, kelem);

    grd = Grid2D.HangingNodes2D(true, false, box1, box2, box3);

    // HIER ANPASSUNG DER RANDBEDINGUNGEN!
    grd.EdgeTagNames.Add(1, "pressure_outlet");

 
    grd.DefineEdgeTags(delegate (double[] X) {
        byte et = 0;
        if (Math.Abs(X[1] - yNodes.Last()) <= 1.0e-8) // UPPER BOUNDARY
            et = 1;
        if (Math.Abs(X[1] - yNodes.First()) <= 1.0e-8) // LOWER BOUNDARY
            et = 1;
        if (Math.Abs(X[0] - xNodes.First()) <= 1.0e-8) // LEFT BOUNDARY
            et = 1;
        if (Math.Abs(X[0] - xNodes.Last()) <= 1.0e-8) // RIGHT BOUNDARY
            et = 1;
        return et;
    });

    // grd.Name = "StaticDropletOnWall_meshStudy";
    grd.Name = "CapillaryWave_meshstudy_" + kelem;        
    
    BoSSSshell.WorkflowMgm.DefaultDatabase.SaveGrid(ref grd);
    
    GridSeq[iGrid] = grd;
}

In [ ]:
public static XNSE_Control CW_forWorksheet() {

    XNSE_Control C = new XNSE_Control();

    //C.DbPath = set by workflowMgm during job creation
    C.savetodb = true;
    C.ContinueOnIoError = false;
    
    // Physical Parameters
    // ===================
    C.PhysicalParameters.IncludeConvection = false;
    C.PhysicalParameters.Material = false;

    // misc. solver options
    // ====================
    C.NonLinearSolver.MaxSolverIterations = 50;
    C.NonLinearSolver.ConvergenceCriterion = 1e-8;
    C.LevelSet_ConvergenceCriterion = 1e-6;

    // Level-Set options (AMR)
    // =======================
    C.LSContiProjectionMethod = BoSSS.Solution.LevelSetTools.ContinuityProjectionOption.ConstrainedDG;
    C.Option_LevelSetEvolution = BoSSS.Solution.LevelSetTools.LevelSetEvolution.StokesExtension;
    C.AdvancedDiscretizationOptions.SST_isotropicMode = BoSSS.Solution.LevelSetTools.SurfaceStressTensor_IsotropicMode.LaplaceBeltrami_ContactLine;

    // Timestepping
    // ============
    C.TimesteppingMode = AppControl._TimesteppingMode.Transient;

    C.TimeSteppingScheme = BoSSS.Solution.XdgTimestepping.TimeSteppingScheme.ImplicitEuler;
    C.Timestepper_BDFinit = BoSSS.Solution.Timestepping.TimeStepperInit.SingleInit;
    C.Timestepper_LevelSetHandling = BoSSS.Solution.XdgTimestepping.LevelSetHandling.LieSplitting;

    C.Endtime = 1000;
    C.saveperiod = 1;       

    return C;

}

public static List<XNSE_Control> CapillaryWave(int[] kS, int[] tratios, int[] pDegS, IGridInfo[] grdS){
    
    List<XNSE_Control> controls = new List<XNSE_Control>();
    foreach(int k in kS){
        foreach(int tratio in tratios){
            foreach(int pDeg in pDegS){
                foreach(IGridInfo grd in grdS){
                    controls.Add(CapillaryWave(k, tratio, pDeg, grd));
                }
            }
        }
    }
    
    return controls;
    
}

public static XNSE_Control CapillaryWave(int k, int tratio, int pDeg, IGridInfo grd) {
    
    var C = CW_forWorksheet(); // ICH HABE DIE WESENTLICHEN EINSTELLUNGEN HIER DIREKT IN DAS WORKSHEET ALS STATISCHE METHODE GESCHRIEBEN!

    C.Paramstudy_CaseIdentification.Add(new Tuple<string, object>("wavenumber", k));
    C.Paramstudy_CaseIdentification.Add(new Tuple<string, object>("degree", pDeg));
    C.Paramstudy_CaseIdentification.Add(new Tuple<string, object>("gridlevel", ((GridCommons)grd).Name.Split('_').Last()));

    C.SetDGdegree(pDeg);        
    C.SetGrid(grd);

    C.PhysicalParameters.rho_A             = 1.0;
    C.PhysicalParameters.rho_B             = 1.0;
    C.PhysicalParameters.mu_A              = 0.0001;
    C.PhysicalParameters.mu_B              = 0.0001;
    C.PhysicalParameters.Sigma             = 1.0;

    C.LinearSolver                  = LinearSolverCode.direct_mumps.GetConfig();
    // C.LinearSolver                  = LinearSolverCode.direct_pardiso.GetConfig();
    
    //VERDAMPFUNG IN MEDIUM A
    C.AddBoundaryValue("pressure_outlet");

//     C.AddInitialValue("Phi", $"X => X[1] - 0.1*Math.Exp(-(X[0]*X[0])/(0.1*0.1))", false); // planar with a smoothed dirac to exite some capillary waves
    C.AddInitialValue("Phi", $"X => X[1] - 0.01/{k}*Math.Sin(2*Math.PI*{k}*X[0])", false); // standing wave as an initial disturbance

    // MAKE TIMESTEPPING SETTINGS MORE CLEAR!
    C.TimeSteppingScheme = BoSSS.Solution.XdgTimestepping.TimeSteppingScheme.ImplicitEuler;
    
    // set timestep, such that the grid N=17 should just obey the capillary timestep restriction:
    double c = Math.Sqrt((2*Math.PI*k*C.PhysicalParameters.Sigma)/(C.PhysicalParameters.rho_A+C.PhysicalParameters.rho_B)); // (undamped) phasevelocity of the standing wave 
    double h = ((GridCommons)grd).GridData.Cells.h_minGlobal;
    double dt =  h/((2*pDeg+1)*c);
    C.dtFixed = dt * (5.0)/(5.0 + tratio); // formerly 10 instead of 5
    C.Endtime = 1.0/(c*k); // two periods
    String.Format($"h: {h}; k: {k}; c: {c}; dt: {C.dtFixed}; t1: {C.Endtime}").Display();
    C.NoOfTimesteps = (int)Math.Ceiling(C.Endtime/C.dtFixed);
    
    C.SessionName = ((GridCommons)grd).Name + //grid
        "_K" + k +
        "_P" + pDeg + // degree 
        "_ti" + tratio; // timestep 
    
    C.Paramstudy_CaseIdentification.Add(new Tuple<string, object>("timeratio", tratio.ToString("N2")));
    
    //C.TracingNamespaces = "*";
    
    return C;
}

In [82]:
int[] kS = new int[] { 1, 2, 3, 4}; // "normalized" wavenumbers, i.e. divided by 2pi \lambda = 1/k
int[] degS = new int[] { 2, 3, 4 };
int[] tratioS = new int[] {0, 1, 2, 3, 4, 5}; // timesteps originally / timesteps new

In [83]:
List<XNSE_Control> controls = new List<XNSE_Control>();

In [84]:
// Testcase setup
// ==============

{
    var ctrls = CapillaryWave(kS, tratioS, degS,GridSeq);
    controls.AddRange(ctrls);
}

Console.WriteLine("# of controls: " + controls.Count());


In [85]:
controls.ForEach(c => Console.WriteLine(c.SessionName));

In [86]:
//// // FOR TESTING IF THIS RUNS
//// foreach(var C in controls){
//var C = controls.ElementAt(0);
//C.ImmediatePlotPeriod = 1;
//C.SuperSampling = 2;
//C.savetodb = false;
//C.PostprocessingModules.Clear();
//using(var solver = new XNSE()){
//    solver.Init(C);
//    solver.RunSolverMode();
//}
// }


In [87]:
bool run      = true;
var bpc = BoSSSshell.GetDefaultQueue();

In [88]:
bpc

In [89]:
if(BoSSSshell.WorkflowMgm.AllJobs.Count() > 0){
     BoSSSshell.WorkflowMgm.ResetProject();
}
var jobs = controls.Select(c => c.CreateJob()).ToArray();
jobs.ForEach(j => j.NumberOfMPIProcs = 1);
jobs.ForEach(j => j.NumberOfThreads = 1);
jobs.ForEach(j => j.EnvironmentVars["BOSSS_ARG_7"] = j.NumberOfThreads.ToString());

In [ ]:
jobs.Activate();

In [ ]:
BoSSSshell.WorkflowMgm.BlockUntilAllJobsTerminate(18000);

In [103]:
var sessions = BoSSSshell.WorkflowMgm.Sessions;
sessions

In [96]:
//var sess = sessions.Where(s => s.SuccessfulTermination);
//var sess = sessions.Where(s => Convert.ToInt32(s.KeysAndQueries["id:degree"]) == 4);
//sess

In [ ]:
//sessions.Skip(53).Take(1).ForEach(s => s.Export().To(Path.GetFullPath(Path.Join(".",BoSSSshell.WorkflowMgm.CurrentProject + "-" + s.Name))).WithShadowFields().WithSupersampling(2).Do());

In [101]:
static void WriteTexTable(this IEnumerable<ISessionInfo> S, int[] kS, int[] degS, int[] iS){
           
    string WriteTableRow(int k){
        string row = "\t\t" + @"$j="+k+"$\n";
        foreach(int i in iS){
            row += WriteTableMultiCell(k, i) + "\n";
        }
        return row;
    }
    
    string WriteTableMultiCell(int k, int i){
        string multcell = "\t\t";
        foreach(int deg in degS){
            multcell += @"&" + WriteTableCell(k, deg, i);
        }
        return multcell;
    }
    
    string WriteTableCell(int k, int deg, int i){
        var ss = S.Where(s => s.Name.Contains("K" + k +"_P" + deg + "_ti" + i)).Single();
        return ss.SuccessfulTermination ? @" \cellcolor{green!25} " : @" \cellcolor{red!25} ";
    }
    
    Directory.CreateDirectory($"{BoSSSshell.wmg.CurrentProject}");
    using(var stw = new StreamWriter($"{BoSSSshell.wmg.CurrentProject}/CapillaryWaveSuccess.txt")) {
        
        stw.WriteLine(@"\begin{table}");
        stw.WriteLine("\t"+@"\centering");
        stw.WriteLine("\t"+@"\caption{Capillary time-step simulation success}");
        stw.WriteLine("\t"+@"\label{tab:capdt1}");
        
        stw.Write("\t"+@"\begin{tabularx}{\linewidth}{ >{\hsize=2.35\hsize\centering\arraybackslash}X");
        for(int i = 0; i < iS.Length; i++){
            stw.Write(@" |");
            for(int deg = 0; deg < degS.Length; deg++){
                stw.Write(@" >{\hsize=0.925\hsize\centering\arraybackslash}X");
            }            
        }
        stw.WriteLine(@"}");
        
        stw.Write("\t\t");
        for(int i = 0; i < iS.Length; i++){
            stw.Write(@" & \multicolumn{"+degS.Length+@"}{|c}{$i="+iS[i]+@"$}");                     
        }
        stw.WriteLine(@"\\");
        
        //stw.Write("\t\t");
        //int i0 = 2;
        //for(int i = 0; i < iS.Length; i++){
        //    stw.Write(@"\cmidrule(lr){"+(i0)+"-"+(i0+degS.Length-1)+@"}"); 
        //    i0 += degS.Length;
        //}
        //stw.WriteLine(@"\\");
        //stw.WriteLine("\t\t" + @"\hline");  
        
        stw.Write("\t\t" + @"$\Pdeg$");
        for(int i = 0; i < iS.Length; i++){
            for(int deg = 0; deg < degS.Length; deg++){
                stw.Write(@" & $"+degS[deg] + "$");
            }            
        }
        stw.WriteLine(@"\\");        

        foreach(int k in kS){
            stw.Write("\t\t" + @"\hline" + "\n" + WriteTableRow(k) + "\t\t" +@"\\" + "\n");
        }     

        stw.WriteLine("\t"+@"\end{tabularx}");
        stw.WriteLine(@"\end{table}");
    }      
}

Export the result (fail/success) as a colored tex table

In [ ]:
//WriteTexTable(sessions, kS, degS, tratioS)

Assertion that the expected number of simulations fail.

In [ ]:
int expectedSuccess = 52; // at the time of the thesis 20 simulations failed!
int success = BoSSSshell.wmg.Sessions.Where(s => s.SuccessfulTermination).Count();

if(success > expectedSuccess){
    throw new ApplicationException("More than the expected number of simulations succeeded! This may be a good thing");
} else if(success < expectedSuccess){
    throw new ApplicationException("Less than the expected number of simulations succeeded!");
}